# 01 - Generar datos sinteticos de ventas

**Todo es ficticio.** No se descargan datos, no se usan secretos y no se instalan paquetes.
Este notebook es autonomo: no necesita clonar el repositorio ni cargar archivos Python.

## Antes de pulsar Ejecutar todo
1. Importa este `.ipynb` en un workspace de Microsoft Fabric con capacidad activa.
2. Crea un Lakehouse vacio para el taller y adjuntalo como **predeterminado**.
3. Usa el kernel PySpark. Si cambias el Lakehouse, reinicia la sesion Spark.
4. Revisa la siguiente celda. `SCHEMA = "dbo"` corresponde a un Lakehouse con esquemas.
   Para un Lakehouse antiguo sin esquemas, usa `SCHEMA = ""`.
5. Ejecutar todo crea **solo** `workshop_products`, `workshop_stores` y
   `workshop_sales_raw`. Si cualquiera existe, se detiene antes de escribir.

**Repeticion:** usa otro Lakehouse o cambia explicitamente `ALLOW_OVERWRITE = True`
despues de revisar el destino. Esto reemplaza esas tres tablas y su esquema.
Las tres escrituras no son una transaccion: un fallo intermedio puede dejar tablas parciales.
No uses un Lakehouse compartido con datos de produccion.


In [ ]:
N_SALES = 10_000
SEED = 42
START_DATE = "2026-01-01"
DAYS = 90
SCHEMA = "dbo"
ALLOW_OVERWRITE = False


## Generador reutilizable
Solo biblioteca estandar; semilla fija y fechas fijas.


In [ ]:
"""Deterministic, entirely fictional retail data; Python standard library only."""

import random
from dataclasses import dataclass
from datetime import date, timedelta
from decimal import ROUND_HALF_UP, Decimal


@dataclass(frozen=True)
class GeneratorConfig:
    seed: int = 42
    n_sales: int = 10_000
    start_date: str = "2026-01-01"
    days: int = 90
    duplicate_rate: float = 0.02
    anomaly_rate: float = 0.01

    def __post_init__(self):
        bounds = {"seed": (0, 2**32 - 1), "n_sales": (100, 100_000), "days": (1, 3660)}
        for field, (low, high) in bounds.items():
            value = getattr(self, field)
            if type(value) is not int or not low <= value <= high:
                raise ValueError(f"{field} debe ser un entero entre {low} y {high}.")
        for field, high in (("duplicate_rate", 0.20), ("anomaly_rate", 0.10)):
            value = getattr(self, field)
            if type(value) not in (int, float) or not 0 <= value <= high:
                raise ValueError(f"{field} debe estar entre 0 y {high}.")
        start = date.fromisoformat(self.start_date)
        if start.isoformat() != self.start_date:
            raise ValueError("start_date debe usar el formato YYYY-MM-DD.")
        start + timedelta(days=self.days - 1)


def generate_data(config):
    rng = random.Random(config.seed)
    categories = ("Hogar", "Deporte", "Tecnologia", "Papeleria")
    products = [
        {
            "product_id": index + 1,
            "product_name": f"Producto {index + 1:02d}",
            "category": categories[index // 3],
            "list_price": Decimal(500 + index * 725) / 100,
        }
        for index in range(12)
    ]
    stores = [
        {"store_id": index + 1, "store_name": f"Tienda {index + 1:02d}", "city": city}
        for index, city in enumerate(("Ciudad Norte", "Ciudad Sur", "Ciudad Este", "Ciudad Oeste"))
    ]
    start = date.fromisoformat(config.start_date)
    sales = []
    for index in range(config.n_sales):
        product = rng.choice(products)
        store = rng.choice(stores)
        sales.append(
            {
                "sale_id": f"V{index + 1:07d}",
                "sale_date": start + timedelta(days=rng.randrange(config.days)),
                "product_id": product["product_id"],
                "store_id": store["store_id"],
                "channel": rng.choice(("online", "tienda")),
                "quantity": rng.randint(1, 5),
                "unit_price": product["list_price"],
                "discount_rate": Decimal(rng.choice((0, 5, 10, 15))) / 100,
                "status": "cancelled" if rng.random() < 0.10 else "completed",
                "category_raw": product["category"],
                "city_raw": store["city"],
            }
        )

    indices = list(range(config.n_sales))
    rng.shuffle(indices)
    count = int(config.n_sales * config.anomaly_rate)
    for kind in range(6):
        for index in indices[kind * count : (kind + 1) * count]:
            row = sales[index]
            if kind == 0:
                row["category_raw"] = None
            elif kind == 1:
                row["city_raw"] = None
            elif kind == 2:
                row["quantity"] = 0 if index % 2 == 0 else -1
            elif kind == 3:
                row["product_id"] = 999_999
            elif kind == 4:
                row["category_raw"] = f"  {row['category_raw'].upper()}  "
            else:
                row["city_raw"] = f"  {row['city_raw'].upper()}  "

    duplicate_count = int(config.n_sales * config.duplicate_rate)
    duplicates = [
        dict(sales[index]) for index in rng.sample(range(config.n_sales), duplicate_count)
    ]
    raw_sales = sales + duplicates
    rng.shuffle(raw_sales)
    return {"products": products, "stores": stores, "sales_raw": raw_sales}


def reference_clean_rows(data):
    """Small Python oracle for the exercise, not a production ETL implementation."""
    products = {row["product_id"]: row for row in data["products"]}
    stores = {row["store_id"]: row for row in data["stores"]}
    seen = {}
    cleaned = []
    for row in data["sales_raw"]:
        sale_id = row["sale_id"]
        if sale_id in seen:
            if row != seen[sale_id]:
                raise ValueError(f"Duplicados con contenido diferente: {sale_id}")
            continue
        seen[sale_id] = row
        product = products.get(row["product_id"])
        store = stores.get(row["store_id"])
        if not product or not store or row["quantity"] <= 0 or row["status"] != "completed":
            continue
        clean = {
            key: value for key, value in row.items() if key not in ("category_raw", "city_raw")
        }
        clean["category"] = product["category"]
        clean["city"] = store["city"]
        clean["net_amount"] = (
            row["quantity"] * row["unit_price"] * (1 - row["discount_rate"])
        ).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        cleaned.append(clean)
    return cleaned


def expected_results(data):
    raw = data["sales_raw"]
    cleaned = reference_clean_rows(data)
    product_ids = {row["product_id"] for row in data["products"]}
    by_category = {}
    for row in cleaned:
        category = row["category"]
        by_category[category] = by_category.get(category, Decimal("0.00")) + row["net_amount"]
    return {
        "products": len(data["products"]),
        "stores": len(data["stores"]),
        "raw_rows": len(raw),
        "unique_sales": len({row["sale_id"] for row in raw}),
        "extra_duplicate_rows": len(raw) - len({row["sale_id"] for row in raw}),
        "null_category_rows": sum(row["category_raw"] is None for row in raw),
        "null_city_rows": sum(row["city_raw"] is None for row in raw),
        "invalid_quantity_rows": sum(row["quantity"] <= 0 for row in raw),
        "orphan_product_rows": sum(row["product_id"] not in product_ids for row in raw),
        "cancelled_rows": sum(row["status"] == "cancelled" for row in raw),
        "clean_rows": len(cleaned),
        "net_revenue": str(sum((row["net_amount"] for row in cleaned), Decimal("0.00"))),
        "revenue_by_category": {key: str(value) for key, value in sorted(by_category.items())},
    }


In [ ]:
"""Fabric notebook helpers. Uses the existing notebook Spark session."""

import re

TABLE_NAMES = {
    "products": "workshop_products",
    "stores": "workshop_stores",
    "sales_raw": "workshop_sales_raw",
}


def table_names(schema):
    if not isinstance(schema, str) or (
        schema and not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
    ):
        raise ValueError("SCHEMA debe ser un identificador simple o una cadena vacia.")
    prefix = f"`{schema}`." if schema else ""
    return {key: f"{prefix}`{value}`" for key, value in TABLE_NAMES.items()}


def require_default_lakehouse(context):
    if not context.get("defaultLakehouseId"):
        raise RuntimeError(
            "Adjunta un Lakehouse como predeterminado al notebook y reinicia la sesion Spark."
        )


def create_dataframes(spark, data):
    from pyspark.sql import types as T

    schemas = {
        "products": T.StructType(
            [
                T.StructField("product_id", T.IntegerType(), False),
                T.StructField("product_name", T.StringType(), False),
                T.StructField("category", T.StringType(), False),
                T.StructField("list_price", T.DecimalType(18, 2), False),
            ]
        ),
        "stores": T.StructType(
            [
                T.StructField("store_id", T.IntegerType(), False),
                T.StructField("store_name", T.StringType(), False),
                T.StructField("city", T.StringType(), False),
            ]
        ),
        "sales_raw": T.StructType(
            [
                T.StructField("sale_id", T.StringType(), False),
                T.StructField("sale_date", T.DateType(), False),
                T.StructField("product_id", T.IntegerType(), False),
                T.StructField("store_id", T.IntegerType(), False),
                T.StructField("channel", T.StringType(), False),
                T.StructField("quantity", T.IntegerType(), False),
                T.StructField("unit_price", T.DecimalType(18, 2), False),
                T.StructField("discount_rate", T.DecimalType(4, 2), False),
                T.StructField("status", T.StringType(), False),
                T.StructField("category_raw", T.StringType(), True),
                T.StructField("city_raw", T.StringType(), True),
            ]
        ),
    }
    return {key: spark.createDataFrame(data[key], schema) for key, schema in schemas.items()}


def write_tables(spark, frames, schema, allow_overwrite=False):
    if type(allow_overwrite) is not bool:
        raise ValueError("ALLOW_OVERWRITE debe ser True o False.")
    names = table_names(schema)
    if set(frames) != set(names):
        raise ValueError("Se requieren exactamente products, stores y sales_raw.")
    if schema:
        spark.sql(f"SHOW TABLES IN `{schema}`").limit(1).collect()
    existing = [name for name in names.values() if spark.catalog.tableExists(name)]
    if existing and not allow_overwrite:
        raise FileExistsError(
            "No se ha escrito ninguna tabla. Ya existen: "
            + ", ".join(existing)
            + ". Usa otro Lakehouse o revisa ALLOW_OVERWRITE."
        )
    mode = "overwrite" if allow_overwrite else "errorifexists"
    for key, name in names.items():
        writer = frames[key].write.format("delta").mode(mode)
        if allow_overwrite:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(name)
    counts = {key: spark.table(name).count() for key, name in names.items()}
    for key, actual in counts.items():
        expected = frames[key].count()
        if actual != expected:
            raise RuntimeError(f"Recuento incorrecto en {names[key]}: {actual} != {expected}")
    return counts


## Generar, inspeccionar y conocer los resultados esperados


In [ ]:
import json

import notebookutils

require_default_lakehouse(notebookutils.runtime.context)
config = GeneratorConfig(seed=SEED, n_sales=N_SALES, start_date=START_DATE, days=DAYS)
names = table_names(SCHEMA)
print("Tablas de destino:", names)
data = generate_data(config)
report = expected_results(data)
frames = create_dataframes(spark, data)
print(json.dumps(report, indent=2, ensure_ascii=False))
display(frames["sales_raw"].orderBy("sale_id").limit(10))


## Guardar las tres tablas Delta en el Lakehouse predeterminado


In [ ]:
saved_counts = write_tables(spark, frames, SCHEMA, allow_overwrite=ALLOW_OVERWRITE)
print("Tablas Delta guardadas y recuentos comprobados:", saved_counts)


## Listo para el taller

Con los parametros por defecto hay 12 productos, 4 tiendas y 10.200 filas de ventas:
10.000 ventas distintas y 200 copias exactas. Se introducen nulos, texto con espacios,
cantidades no validas y productos inexistentes. Las cancelaciones son otro estado de
negocio, no un error de calidad.

Los contadores del informe incluyen duplicados cuando el nombre termina en `_rows`.
No sumes incidencias para calcular ventas rechazadas: pueden solaparse.

**Reto:** pide a Copilot un perfilado y una propuesta de limpieza antes de ejecutar
cambios. Recupera categoria y ciudad desde las dimensiones; no elimines esas ventas
solo porque los campos raw sean nulos. Redondea cada linea a dos decimales ANTES
de sumar. La solucion de referencia esta separada para no desvelar el ejercicio.

El SQL analytics endpoint puede tardar en mostrar las tablas. Refresca el explorador
y confirma el esquema antes de ejecutar los ejemplos T-SQL del repositorio.
Deten la sesion Spark desde la interfaz de Fabric al terminar.
